# Milestone 2: Digit classifier from scratch and in PyTorch
*AI from scratch, milestone 2* · combine notebook for Stage 2

Build a handwritten digit recognizer (0 to 9) from the ground up:
1. Forward pass, cross-entropy loss, and backpropagation implemented directly in NumPy without autograd.
2. Gradient descent training loop reaching over 85% accuracy.
3. Re-implementing the exact same architecture in PyTorch with `nn.Sequential` and autograd.
4. Verifying that both models learn equivalent representations.

Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Compute z1 = X @ W1 + b1, a1 = np.maximum(0, z1), z2 = a1 @ W2 + b2', 'exp_z = np.exp(z2 - np.max(z2, axis=1, keepdims=True)); p = exp_z / np.sum(exp_z, axis=1, keepdims=True)'], 2: ['eps = 1e-12; return -float(np.mean(np.sum(y_oh * np.log(p + eps), axis=1)))'], 3: ['N = len(X); dz2 = (p - y_oh) / N; dW2 = a1.T @ dz2; db2 = np.sum(dz2, axis=0)', 'da1 = dz2 @ W2.T; dz1 = da1 * (z1 > 0); dW1 = X.T @ dz1; db1 = np.sum(dz1, axis=0)'], 4: ['In a loop for 60 epochs: z1, a1, z2, p = forward(X_tr, W1, b1, W2, b2); dW1, db1, dW2, db2 = backward(...); update W1, b1, W2, b2'], 5: ['pt_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))'], 6: ['criterion = nn.CrossEntropyLoss(); opt = torch.optim.SGD(pt_model.parameters(), lr=0.5)', 'for 60 epochs: opt.zero_grad(); loss = criterion(pt_model(t_Xtr), t_ytr); loss.backward(); opt.step()']}
_ANSWERS = {1: 'ejEgPSBYIEAgVzEgKyBiMQphMSA9IG5wLm1heGltdW0oMCwgejEpCnoyID0gYTEgQCBXMiArIGIyCmV4cF96ID0gbnAuZXhwKHoyIC0gbnAubWF4KHoyLCBheGlzPTEsIGtlZXBkaW1zPVRydWUpKQpwID0gZXhwX3ogLyBucC5zdW0oZXhwX3osIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKcmV0dXJuIHoxLCBhMSwgejIsIHA=', 2: 'ZXBzID0gMWUtMTIKcmV0dXJuIC1mbG9hdChucC5tZWFuKG5wLnN1bSh5X29oICogbnAubG9nKHAgKyBlcHMpLCBheGlzPTEpKSk=', 3: 'TiA9IGxlbihYKQpkejIgPSAocCAtIHlfb2gpIC8gTgpkVzIgPSBhMS5UIEAgZHoyCmRiMiA9IG5wLnN1bShkejIsIGF4aXM9MCkKZGExID0gZHoyIEAgVzIuVApkejEgPSBkYTEgKiAoejEgPiAwKQpkVzEgPSBYLlQgQCBkejEKZGIxID0gbnAuc3VtKGR6MSwgYXhpcz0wKQpyZXR1cm4gZFcxLCBkYjEsIGRXMiwgZGIy', 4: 'Zm9yIGVwb2NoIGluIHJhbmdlKDYwKToKICAgIHoxLCBhMSwgejIsIHAgPSBmb3J3YXJkKFhfdHIsIFcxLCBiMSwgVzIsIGIyKQogICAgZFcxLCBkYjEsIGRXMiwgZGIyID0gYmFja3dhcmQoWF90ciwgeV90cl9vaCwgejEsIGExLCBwLCBXMikKICAgIFcxIC09IGxyICogZFcxCiAgICBiMSAtPSBsciAqIGRiMQogICAgVzIgLT0gbHIgKiBkVzIKICAgIGIyIC09IGxyICogZGIyCgpfLCBfLCBfLCBwX3RlID0gZm9yd2FyZChYX3RlLCBXMSwgYjEsIFcyLCBiMikKbnVtcHlfYWNjID0gZmxvYXQoKHBfdGUuYXJnbWF4KGF4aXM9MSkgPT0geV90ZSkubWVhbigpKQ==', 5: 'cHRfbW9kZWwgPSBubi5TZXF1ZW50aWFsKAogICAgbm4uTGluZWFyKDY0LCAzMiksCiAgICBubi5SZUxVKCksCiAgICBubi5MaW5lYXIoMzIsIDEwKQop', 6: 'Y3JpdGVyaW9uID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpCm9wdCA9IHRvcmNoLm9wdGltLlNHRChwdF9tb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTAuNSkKCmZvciBlcG9jaCBpbiByYW5nZSg2MCk6CiAgICBvcHQuemVyb19ncmFkKCkKICAgIGxvc3MgPSBjcml0ZXJpb24ocHRfbW9kZWwodF9YdHIpLCB0X3l0cikKICAgIGxvc3MuYmFja3dhcmQoKQogICAgb3B0LnN0ZXAoKQoKd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICBwdF9wcmVkcyA9IHB0X21vZGVsKHRfWHRlKS5hcmdtYXgoZGltPTEpCiAgICBweXRvcmNoX2FjYyA9IGZsb2F0KChwdF9wcmVkcyA9PSB0X3l0ZSkuZmxvYXQoKS5tZWFuKCkuaXRlbSgpKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Load 1,797 8x8 digit images (values 0..16)
X_raw, y = load_digits(return_X_y=True)
X = X_raw / 16.0  # Normalize to [0, 1]
y_onehot = np.eye(10)[y]

# Train (1,400) / Validation (397) split
tr, te = np.arange(1400), np.arange(1400, len(X))
X_tr, y_tr, y_tr_oh = X[tr], y[tr], y_onehot[tr]
X_te, y_te, y_te_oh = X[te], y[te], y_onehot[te]

print(f"Data ready: {len(X_tr)} train images, {len(X_te)} test images. Input shape: {X.shape[1]}")

## 1. Forward pass in NumPy
We use a 2-layer network: input (64) → hidden (32) with ReLU → output (10) with Softmax.
Write `forward(X, W1, b1, W2, b2)`:
- $z_1 = X W_1 + b_1$
- $a_1 = \max(0, z_1)$ (ReLU)
- $z_2 = a_1 W_2 + b_2$
- $p = \text{softmax}(z_2)$ (subtract max for numerical stability)
Return `(z1, a1, z2, p)`.

In [ ]:
def forward(X, W1, b1, W2, b2):
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

In [ ]:
_W1, _b1 = np.random.randn(64, 32) * 0.05, np.zeros(32)
_W2, _b2 = np.random.randn(32, 10) * 0.05, np.zeros(10)
check(1, lambda: forward(X_tr[:5], _W1, _b1, _W2, _b2)[3].shape == (5, 10) and np.allclose(forward(X_tr[:5], _W1, _b1, _W2, _b2)[3].sum(axis=1), 1.0), "forward pass returns valid probability distributions summing to 1")

## 2. Cross-entropy loss
Write `compute_loss(p, y_oh)` computing categorical cross-entropy:
$$L = -\frac{1}{N} \sum_{i=1}^N \sum_{k=1}^{10} y_{ik} \log(p_{ik} + \epsilon)$$
Use $\epsilon = 10^{-12}$ to prevent $\log(0)$.

In [ ]:
def compute_loss(p, y_oh):
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

In [ ]:
check(2, lambda: np.isclose(compute_loss(forward(X_tr[:5], _W1, _b1, _W2, _b2)[3], y_tr_oh[:5]), np.log(10), atol=0.5), "initial cross-entropy on random weights is approximately ln(10) ≈ 2.30")

## 3. Backpropagation: computing gradients
Write `backward(X, y_oh, z1, a1, p, W2)`:
- $dz_2 = \frac{1}{N} (p - y_{\text{oh}})$
- $dW_2 = a_1^\top dz_2$, $db_2 = \sum dz_2$ (down rows)
- $da_1 = dz_2 W_2^\top$
- $dz_1 = da_1 \cdot (z_1 > 0)$ (ReLU derivative)
- $dW_1 = X^\top dz_1$, $db_1 = \sum dz_1$
Return `(dW1, db1, dW2, db2)`.

In [ ]:
def backward(X, y_oh, z1, a1, p, W2):
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...

In [ ]:
check(3, lambda: backward(X_tr[:5], y_tr_oh[:5], forward(X_tr[:5], _W1, _b1, _W2, _b2)[0], forward(X_tr[:5], _W1, _b1, _W2, _b2)[1], forward(X_tr[:5], _W1, _b1, _W2, _b2)[3], _W2)[0].shape == (64, 32), "gradient matrices match layer weight and bias shapes")

## 4. Training from scratch: Gradient Descent
Train the network for 60 epochs with learning rate 0.5.
Initialize $W_1 \sim \mathcal{N}(0, 0.1^2)$, $b_1 = 0$, $W_2 \sim \mathcal{N}(0, 0.1^2)$, $b_2 = 0$.
Store the final test accuracy on `X_te` in `numpy_acc`.

In [ ]:
W1 = np.random.randn(64, 32) * 0.1
b1 = np.zeros(32)
W2 = np.random.randn(32, 10) * 0.1
b2 = np.zeros(10)
lr = 0.5

# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

if "numpy_acc" in locals() and numpy_acc is not None:
    print(f"NumPy scratch model test accuracy: {numpy_acc * 100:.1f}%")

In [ ]:
check(4, lambda: numpy_acc > 0.85, "NumPy scratch model achieves > 85% test accuracy on handwritten digits")

## 5. The same model in PyTorch
Define an identical network using PyTorch's `nn.Sequential`:
`Linear(64, 32)` → `ReLU()` → `Linear(32, 10)`.
Assign to `pt_model`.

In [ ]:
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...

In [ ]:
check(5, lambda: isinstance(pt_model, nn.Sequential) and len(pt_model) == 3 and pt_model[0].in_features == 64 and pt_model[2].out_features == 10, "PyTorch model architecture matches the 64 -> 32 -> 10 network")

## 6. Training with PyTorch autograd
Convert `X_tr`, `y_tr`, `X_te`, `y_te` to PyTorch tensors (`float32` for features, `int64` for labels).
Train `pt_model` with `nn.CrossEntropyLoss()` and `torch.optim.SGD(pt_model.parameters(), lr=0.5)` for 60 epochs.
Store the final test accuracy on `X_te` in `pytorch_acc`.

In [ ]:
t_Xtr = torch.tensor(X_tr, dtype=torch.float32)
t_ytr = torch.tensor(y_tr, dtype=torch.int64)
t_Xte = torch.tensor(X_te, dtype=torch.float32)
t_yte = torch.tensor(y_te, dtype=torch.int64)

# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

if "pytorch_acc" in locals() and pytorch_acc is not None:
    print(f"PyTorch model test accuracy: {pytorch_acc * 100:.1f}%")

In [ ]:
check(6, lambda: pytorch_acc > 0.85, "PyTorch model reaches > 85% test accuracy, matching the NumPy implementation")